# Notebook 2: Engineered Geometric Features

This notebook evaluates the **Zonal / Edge_arc_feat** ablation for:
- `ArGEnT_self_att_noSDF`
- `PointNetMLPJoint_headfeat`
- `PointNetMLPJoint_FP_headfeat`

All outputs are labelled **validation-split evaluation**: a deterministic 20% geometry holdout (`seed=42`) shared across all models, without proof of full independence from checkpoint selection.


In [ ]:
from __future__ import annotations
import ast, hashlib, importlib.util, inspect, itertools, json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
try:
    import torch
    import h5py
except ImportError as exc:
    raise RuntimeError('Install torch and h5py in the selected notebook kernel before executing this comparison.') from exc
CURRENT_DIR = Path.cwd()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / 'Uniform').exists() else CURRENT_DIR.parent
if not (REPO_ROOT / 'Uniform').exists(): raise RuntimeError(f'Repository root not found from {CURRENT_DIR}')
COMPARISON_DIR = REPO_ROOT / 'Comparison'
RESULTS_DIR = COMPARISON_DIR / 'results' / '02_engineered_geometric_features'
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(COMPARISON_DIR))
import eval_helpers as eh
SPLIT_SEED, EVAL_FRACTION = 42, 0.20
REGIME = 'Zonal'
ABLATION = 'Edge_arc_feat'
FAMILIES = ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint_headfeat', 'PointNetMLPJoint_FP_headfeat']
DATASET_PATH = REPO_ROOT / 'Data_gen' / 'output' / 'disc_dataset_edge_deriv_zonal.h5'
BASELINE_RESULTS_DIR = COMPARISON_DIR / 'results' / '01_fp_vs_argent'
FEATURE_LABELS = {
    0: 'x_mm',
    1: 'r_mm',
    2: 'zone_id',
    3: 'arc_length_mm',
    4: 'tangent_x',
    5: 'tangent_r',
    6: 'curvature',
    7: 'curvature_gradient',
}
COMMIT = __import__('subprocess').check_output(['git','rev-parse','HEAD'], cwd=REPO_ROOT, text=True).strip()
display(Markdown(
    f'**Inspected commit:** `{COMMIT}`\n\n'
    f'**Results directory:** `{RESULTS_DIR}`\n\n'
    f'**Evaluation label:** `validation-split evaluation`'
))


## Checkpoint discovery, script metadata, and feature usage

The discovery step validates checkpoint integrity, required normalization metadata, and head-feature metadata. It also parses the colocated training script so the notebook can compare checkpoint metadata against the declared ablation configuration.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()

def decode(value):
    return value.decode() if isinstance(value, bytes) else value

def json_ready(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value

def compact_hash(obj):
    payload = json.dumps(json_ready(obj), sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()[:16]

def parse_script_metadata(script_path: Path) -> dict:
    meta = {'path': str(script_path)}
    tree = ast.parse(script_path.read_text(encoding='utf-8'))
    wanted = {
        'INPUT_COLS', 'EXTRA_FEAT_COLS', 'HEAD_FEAT_COLS', 'TARGET_NAMES',
        'EXPECTED_REPR', 'H5_FILENAME', 'Perc_training_data', 'train_data_percent'
    }
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in wanted:
                    try:
                        meta[target.id] = ast.literal_eval(node.value)
                    except Exception:
                        pass
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id in wanted:
            try:
                meta[node.target.id] = ast.literal_eval(node.value)
            except Exception:
                pass
        elif isinstance(node, ast.Call) and getattr(node.func, 'id', None) == 'train_test_split':
            for kw in node.keywords:
                if kw.arg in {'test_size', 'random_state'}:
                    try:
                        meta[kw.arg] = ast.literal_eval(kw.value)
                    except Exception:
                        pass
    return meta

def expected_encoder_cols(family: str, payload: dict, script_meta: dict) -> list[int]:
    if family.startswith('ArGEnT'):
        cols = (payload.get('arch') or {}).get('input_cols') or script_meta.get('INPUT_COLS') or [0, 1]
        return [int(c) for c in cols]
    cols = payload.get('extra_feat_cols') or script_meta.get('EXTRA_FEAT_COLS') or []
    return [int(c) for c in cols]

def expected_head_cols(family: str, payload: dict, script_meta: dict) -> list[int]:
    cols = payload.get('head_feat_cols', payload.get('headfeatcols'))
    if cols is None:
        cols = script_meta.get('HEAD_FEAT_COLS') or []
    return [int(c) for c in cols]

def discover_checkpoints():
    rows = []
    for family in FAMILIES:
        folder = REPO_ROOT / REGIME / ABLATION / family
        checkpoints = sorted((folder / 'Trained_models').glob('*.pt'))
        scripts = sorted(folder.glob('Training_script*.py'))
        script_meta = parse_script_metadata(scripts[0]) if scripts else {}
        if not checkpoints:
            rows.append({
                'regime': REGIME,
                'ablation': ABLATION,
                'model_family': family,
                'status': 'missing checkpoint',
                'checkpoint_path': None,
                'training_scripts': [str(p) for p in scripts],
            })
            continue
        for path in checkpoints:
            status = 'discovered'
            metadata = {}
            try:
                payload = torch.load(path, map_location='cpu', weights_only=False)
                required = ['arch', 'model_state', 'coord_center', 'coord_half_range', 'target_mean', 'target_std']
                missing = [k for k in required if k not in payload]
                if missing:
                    status = 'incompatible: missing ' + ', '.join(missing)
                if family.endswith('headfeat') and not (payload.get('head_feat_cols', payload.get('headfeatcols'))):
                    status = 'incompatible: missing head-feature metadata'
                metadata = {
                    'arch': json_ready(payload.get('arch')),
                    'arch_hash': compact_hash(payload.get('arch')) if payload.get('arch') is not None else None,
                    'model_name': payload.get('model_name'),
                    'target_names': json_ready(payload.get('target_names')),
                    'extra_feat_cols': json_ready(payload.get('extra_feat_cols')),
                    'head_feat_cols': json_ready(payload.get('head_feat_cols', payload.get('headfeatcols'))),
                    'encoder_input_cols': expected_encoder_cols(family, payload, script_meta),
                    'head_input_cols': expected_head_cols(family, payload, script_meta),
                    'script_meta': json_ready(script_meta),
                    'script_repr': script_meta.get('EXPECTED_REPR'),
                    'script_h5_filename': script_meta.get('H5_FILENAME'),
                }
                if script_meta.get('EXPECTED_REPR') not in (None, 'edge'):
                    status = f"incompatible: training script expects representation {script_meta.get('EXPECTED_REPR')!r}"
                if script_meta.get('H5_FILENAME') not in (None, DATASET_PATH.name):
                    status = f"incompatible: training script points to {script_meta.get('H5_FILENAME')!r}"
            except Exception as exc:
                status = 'incompatible: ' + type(exc).__name__ + ': ' + str(exc)
            rows.append({
                'regime': REGIME,
                'ablation': ABLATION,
                'model_family': family,
                'status': status,
                'checkpoint_path': str(path),
                'file_size_bytes': path.stat().st_size,
                'sha256': sha256(path),
                'training_scripts': [str(p) for p in scripts],
                **metadata,
            })
    report = pd.DataFrame(rows)
    report.to_json(RESULTS_DIR / 'checkpoint_integrity.json', orient='records', indent=2)
    return report

checkpoint_report = discover_checkpoints()
display(checkpoint_report[[
    'model_family', 'status', 'checkpoint_path', 'file_size_bytes', 'arch_hash',
    'extra_feat_cols', 'head_feat_cols'
]])


In [ ]:
def names_for_cols(cols):
    return [FEATURE_LABELS.get(int(c), f'feature_col_{c}') for c in (cols or [])]

feature_rows = []
for family in FAMILIES:
    sub = checkpoint_report[(checkpoint_report.model_family == family) & (checkpoint_report.status == 'discovered')]
    if sub.empty:
        feature_rows.append({
            'model_family': family,
            'encoder_full_columns': None,
            'encoder_inputs': None,
            'encoder_engineered_features': None,
            'head_columns': None,
            'head_inputs': None,
            'note': 'checkpoint unavailable',
        })
        continue
    row = sub.iloc[0]
    encoder_cols = row.encoder_input_cols or []
    head_cols = row.head_input_cols or []
    engineered_encoder = [c for c in encoder_cols if int(c) >= 3]
    note = 'Head uses explicit geometric features.' if head_cols else 'No explicit head features.'
    if family.startswith('ArGEnT'):
        note = 'Cross-attention encoder receives explicit geometric features; head query remains (x, r) only.'
    feature_rows.append({
        'model_family': family,
        'encoder_full_columns': encoder_cols,
        'encoder_inputs': ', '.join(names_for_cols(encoder_cols)),
        'encoder_engineered_features': ', '.join(names_for_cols(engineered_encoder)) or '(none)',
        'head_columns': head_cols,
        'head_inputs': ', '.join(names_for_cols(head_cols)) or '(x, r only)',
        'note': note,
    })
feature_assignment = pd.DataFrame(feature_rows)
eh.save_table(feature_assignment, RESULTS_DIR, 'feature_assignment')
display(feature_assignment)


## HDF5 loading and fixed evaluation split

The notebook re-derives the same deterministic 80/20 geometry split inside the notebook and records the provenance as **validation-split evaluation**.


In [ ]:
def load_samples(path):
    samples = []
    if not path.exists():
        raise FileNotFoundError(f'Dataset not found: {path}')
    with h5py.File(path, 'r') as h5:
        representation = decode(h5.attrs.get('representation', ''))
        if representation != 'edge': raise ValueError(f'{path.name}: expected representation edge, got {representation!r}')
        node_feature_names = [decode(x) for x in np.asarray(h5.get('node_feature_names', []))] if 'node_feature_names' in h5 else []
        for key in sorted(h5['samples'].keys()):
            g = h5['samples'][key]
            def arr(name, default=None): return np.asarray(g[name]) if name in g else default
            coords = arr('node_coords_mm'); stress = arr('stress_max_vm'); life = arr('life_raw')
            if coords is None or stress is None or life is None: raise ValueError(f'{path.name}/{key}: missing required target fields')
            sample_id = decode(g.attrs.get('sample_id', key))
            attrs = {str(k): decode(v) for k, v in g.attrs.items()}
            local_names = [decode(x) for x in np.asarray(g['node_feature_names'])] if 'node_feature_names' in g else node_feature_names
            arc_length = arr('arc_length_mm', np.arange(len(coords), dtype='float32')).reshape(-1).astype('float32')
            raw_node_features = arr('node_features', np.empty((len(coords), 0), dtype='float32')).astype('float32')
            engineered = np.column_stack([arc_length, raw_node_features]).astype('float32')
            samples.append({
                'sample_key': key,
                'sample_id': str(sample_id),
                'attrs': attrs,
                'coords': coords.astype('float32'),
                'stress': stress.astype('float32').reshape(-1),
                'loglife': np.log10(np.clip(life.astype('float64').reshape(-1), 1e-30, None)).astype('float32'),
                'zone_id': arr('zone_id', np.full(len(coords), -1)).reshape(-1),
                'subzone_id': arr('subzone_id', np.full(len(coords), np.nan)).reshape(-1),
                'arc_length_mm': arc_length,
                'node_features': engineered,
                'node_feature_names': ['arc_length_mm', *local_names],
                'raw_node_feature_names': local_names,
            })
    return samples

def split_samples(samples):
    rng = np.random.default_rng(SPLIT_SEED); order = rng.permutation(len(samples)); n_eval = max(1, int(round(len(samples) * EVAL_FRACTION)))
    eval_pos = np.sort(order[:n_eval]).tolist(); train_pos = np.sort(order[n_eval:]).tolist()
    return train_pos, eval_pos

all_samples = load_samples(DATASET_PATH)
train_pos, eval_pos = split_samples(all_samples)
split_record = {
    'regime': REGIME,
    'ablation': ABLATION,
    'dataset_path': str(DATASET_PATH),
    'total_geometry_count': len(all_samples),
    'training_geometry_count': len(train_pos),
    'evaluation_geometry_count': len(eval_pos),
    'training_sample_ids': [all_samples[i]['sample_id'] for i in train_pos],
    'evaluation_sample_ids': [all_samples[i]['sample_id'] for i in eval_pos],
    'split_seed': SPLIT_SEED,
    'split_fraction': EVAL_FRACTION,
    'evaluation_label': 'validation-split evaluation',
    'independence_basis': 'A deterministic geometry holdout is used, but checkpoint-selection independence is not proved.',
    'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'notebook_commit_sha': COMMIT,
}
with open(RESULTS_DIR / 'evaluation_split_provenance.json', 'w', encoding='utf-8') as stream:
    json.dump(split_record, stream, indent=2)
display(pd.DataFrame([{
    'regime': REGIME,
    'ablation': ABLATION,
    'total_geometries': len(all_samples),
    'evaluation_geometries': len(eval_pos),
    'label': 'validation-split evaluation',
}]))


## Model reconstruction and shared-geometry inference

The inference path rebuilds the saved architecture, validates feature availability and normalization metadata, and then filters to the intersection of evaluation geometries successfully predicted by **all three models**.


In [ ]:
def import_local(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def reconstruct(row):
    folder = Path(row['checkpoint_path']).parent.parent; family = row['model_family']
    ckpt = torch.load(row['checkpoint_path'], map_location='cpu', weights_only=False); arch = dict(ckpt['arch'])
    pn = import_local(folder / 'pn_models.py', f"pn_{folder.parent.name}_{folder.name}_{family}")
    sys.modules['pn_models'] = pn
    if family in ('PointNetMLPJoint_FP', 'PointNetMLPJoint_FP_headfeat'):
        if not hasattr(pn, 'build_fp_model_from_arch'): raise RuntimeError('FP checkpoint requires build_fp_model_from_arch')
        model = pn.build_fp_model_from_arch(arch)
    elif family in ('PointNetMLPJoint', 'PointNetMLPJoint_weighted', 'PointNetMLPJoint_headfeat'):
        model = pn.build_model_from_arch(arch)
    else:
        bench = import_local(folder / 'benchmarks.py', f'bench_{family}')
        if not hasattr(bench, 'ArGEnTDeepONet'): raise RuntimeError('No ArGEnTDeepONet found in own benchmarks.py')
        defaults = {'hidden_dim': 128, 'num_heads': 4, 'num_layers': 2, 'output_dim': 128, 'out_channels': 1, 'attention_type': 'self', 'use_sdf': False, 'in_ch_geom': 2}
        defaults.update({k: v for k, v in arch.items() if k in defaults})
        if 'out_channels' not in arch and 'bias' in ckpt['model_state']: defaults['out_channels'] = int(ckpt['model_state']['bias'].shape[0])
        model = bench.ArGEnTDeepONet(**defaults)
    model.load_state_dict(ckpt['model_state'], strict=True)
    model.eval()
    return model, ckpt

def normalization_vectors(ckpt, cols):
    cols = [int(c) for c in (cols or [])]
    stats = ckpt.get('extra_feat_stats')
    if cols and stats is None:
        raise ValueError('missing extra_feat_stats')
    means, stds = [], []
    for i, col in enumerate(cols):
        col_key = int(col)
        if isinstance(stats, dict) and col_key in stats:
            entry = stats[col_key]
            if isinstance(entry, dict):
                means.append(float(entry.get('mean', 0.0)))
                stds.append(float(entry.get('std', 1.0)))
            else:
                means.append(float(entry[0]))
                stds.append(float(entry[1]))
        elif isinstance(stats, dict) and str(col_key) in stats:
            entry = stats[str(col_key)]
            if isinstance(entry, dict):
                means.append(float(entry.get('mean', 0.0)))
                stds.append(float(entry.get('std', 1.0)))
            else:
                means.append(float(entry[0]))
                stds.append(float(entry[1]))
        elif isinstance(stats, (list, tuple, np.ndarray)):
            means.append(float(stats[0][i]))
            stds.append(float(stats[1][i]))
        else:
            means.append(0.0)
            stds.append(1.0)
    return np.asarray(means, dtype='float32'), np.maximum(np.asarray(stds, dtype='float32'), 1e-8)

def feature_matrix(sample, ckpt):
    extra_feat_cols = ckpt.get('extra_feat_cols', []) or []
    available = sample['node_features']
    center = np.asarray(ckpt['coord_center'], dtype='float32')
    half = np.asarray(ckpt['coord_half_range'], dtype='float32')
    coords_norm = (sample['coords'] - center) / np.maximum(half, 1e-8)
    if extra_feat_cols:
        if available.shape[1] < len(extra_feat_cols):
            raise ValueError(f"missing required extra-feature columns: need {len(extra_feat_cols)}, have {available.shape[1]}")
        feat_raw = available[:, :len(extra_feat_cols)].astype('float32')
        means, stds = normalization_vectors(ckpt, extra_feat_cols)
        feat_norm = (feat_raw - means) / stds
    else:
        feat_norm = np.empty((len(sample['coords']), 0), dtype='float32')
    return coords_norm.astype('float32'), feat_norm.astype('float32')

def decode_prediction(out, ckpt):
    out = out.detach().cpu().numpy()
    if out.ndim != 3 or out.shape[0] != 1:
        raise ValueError(f'prediction shape unexpected: {out.shape}')
    mean = np.asarray(ckpt['target_mean'])
    std = np.asarray(ckpt['target_std'])
    out = out * std + mean
    if out.shape[2] == 2: return out[0, :, 0], out[0, :, 1]
    elif out.shape[2] == 1: return np.zeros(out.shape[1], dtype='float32'), out[0, :, 0]
    raise ValueError(f'unexpected output channels: {out.shape[2]}')

def predict_argent(model, sample, ckpt):
    coords_norm, feat_norm = feature_matrix(sample, ckpt)
    x = torch.from_numpy(coords_norm[None])
    q = x.clone()
    if feat_norm.shape[1] > 0:
        geom = torch.cat([x, torch.from_numpy(feat_norm[None])], dim=-1)
    else:
        geom = x
    with torch.no_grad():
        try:
            out = model(geom, q)
        except TypeError:
            out = model(x, q)
    return decode_prediction(out, ckpt)

def predict_headfeat(model, sample, ckpt, family):
    extra_feat_cols = ckpt.get('extra_feat_cols', []) or []
    head_feat_cols = ckpt.get('head_feat_cols', []) or []
    available = sample['node_features']
    center = np.asarray(ckpt['coord_center'], dtype='float32')
    half = np.asarray(ckpt['coord_half_range'], dtype='float32')
    coords_norm = (sample['coords'] - center) / np.maximum(half, 1e-8)
    if extra_feat_cols:
        stats = ckpt.get('extra_feat_stats')
        if stats is None:
            raise ValueError('missing extra_feat_stats')
        if available.shape[1] < len(extra_feat_cols):
            raise ValueError(f"missing required extra-feature columns: need {len(extra_feat_cols)}, have {available.shape[1]}")
        feat_raw = available[:, :len(extra_feat_cols)].astype('float32')
        means, stds = normalization_vectors(ckpt, extra_feat_cols)
        feat_norm = (feat_raw - means) / stds
    else:
        feat_norm = np.empty((len(sample['coords']), 0), dtype='float32')
    if head_feat_cols:
        if available.shape[1] < len(head_feat_cols):
            raise ValueError(f"missing required head-feature columns: need {len(head_feat_cols)}, have {available.shape[1]}")
        head_raw = available[:, :len(head_feat_cols)].astype('float32')
        means_h, stds_h = normalization_vectors(ckpt, head_feat_cols)
        head_norm = (head_raw - means_h) / stds_h
    else:
        head_norm = np.empty((len(sample['coords']), 0), dtype='float32')
    x = torch.from_numpy(coords_norm.astype('float32')[None])
    q = x.clone()
    gf = torch.from_numpy(feat_norm[None]) if feat_norm.shape[1] > 0 else None
    hf = torch.from_numpy(head_norm[None]) if head_norm.shape[1] > 0 else None
    with torch.no_grad():
        if family == 'PointNetMLPJoint_FP_headfeat':
            out = model(x, q, geom_feats=gf, headfeats=hf)
        elif family == 'PointNetMLPJoint_headfeat':
            out = model(x, q, geom_feats=gf, head_feats=hf)
        else:
            raise ValueError(f'Unknown headfeat family: {family}')
    return decode_prediction(out, ckpt)

def predict_dispatch(model, sample, ckpt, family):
    if family == 'ArGEnT_self_att_noSDF':
        return predict_argent(model, sample, ckpt)
    if family in ('PointNetMLPJoint_headfeat', 'PointNetMLPJoint_FP_headfeat'):
        extra_feat_cols = [int(c) for c in (ckpt.get('extra_feat_cols', []) or [])]
        head_feat_cols = [int(c) for c in (ckpt.get('head_feat_cols', []) or [])]
        if extra_feat_cols != [3, 4, 5, 6, 7] or head_feat_cols != [3, 4, 5, 6, 7]:
            raise ValueError(f'unexpected headfeat feature configuration: extra={extra_feat_cols}, head={head_feat_cols}')
        return predict_headfeat(model, sample, ckpt, family)
    raise ValueError(f'Unexpected family in notebook 02: {family}')

node_frames, load_errors, coverage_rows = [], [], []
for _, row in checkpoint_report.iterrows():
    if row.status != 'discovered':
        continue
    try:
        model, ckpt = reconstruct(row)
        expected = ckpt.get('target_names', ['Stress', 'LogLife'])
        if len(expected) != 2:
            raise ValueError(f'incompatible target dimensions: {expected}')
        predicted_ids = []
        for i in eval_pos:
            s = all_samples[i]
            pred_stress, pred_loglife = predict_dispatch(model, s, ckpt, row.model_family)
            predicted_ids.append(s['sample_id'])
            base = pd.DataFrame({
                'regime': REGIME,
                'ablation': ABLATION,
                'model_family': row.model_family,
                'sample_key': s['sample_key'],
                'sample_id': s['sample_id'],
                'node_idx': np.arange(len(s['coords'])),
                'x_mm': s['coords'][:, 0],
                'r_mm': s['coords'][:, 1],
                'zone_id': s['zone_id'],
                'subzone_id': s['subzone_id'],
                'arc_length_mm': s['arc_length_mm'],
                'true_stress': s['stress'],
                'pred_stress': pred_stress,
                'true_loglife': s['loglife'],
                'pred_loglife': pred_loglife,
            })
            base['zone_name'] = base.zone_id.map(eh.ZONE_ID_TO_NAME)
            base['subzone_name'] = base.subzone_id.map(eh.SUBZONE_ID_TO_NAME)
            node_frames.append(base)
        coverage_rows.append({'model_family': row.model_family, 'successful_eval_geometries': len(predicted_ids), 'sample_ids': predicted_ids})
    except Exception as exc:
        load_errors.append({'model_family': row.model_family, 'checkpoint_path': row.checkpoint_path, 'status': 'load/inference failed: ' + repr(exc)})

nodes = pd.concat(node_frames, ignore_index=True) if node_frames else pd.DataFrame()
pd.DataFrame(load_errors).to_json(RESULTS_DIR / 'inference_errors.json', orient='records', indent=2)
coverage = pd.DataFrame(coverage_rows)
if not coverage.empty:
    shared_ids = sorted(set.intersection(*[set(ids) for ids in coverage['sample_ids']]))
else:
    shared_ids = []
if not shared_ids:
    raise RuntimeError('No shared evaluation geometries were successfully predicted by all discovered models.')
coverage_summary = pd.DataFrame([
    {
        'model_family': fam,
        'successful_eval_geometries': len(ids),
        'shared_eval_geometries': len(shared_ids),
        'dropped_to_enforce_fairness': len(ids) - len(shared_ids),
    }
    for fam, ids in zip(coverage['model_family'], coverage['sample_ids'])
])
coverage_summary['evaluation_label'] = 'validation-split evaluation'
eh.save_table(coverage_summary, RESULTS_DIR, 'evaluation_geometry_coverage')
nodes = nodes[nodes.sample_id.isin(shared_ids)].copy()
display(pd.DataFrame(load_errors) if load_errors else coverage_summary)


## Metrics, figures, and paired comparisons


In [ ]:
pooled = eh.pooled_metrics_from_nodes(nodes)
bins = eh.loglife_bin_metrics(nodes)
zones = eh.zone_metrics_from_nodes(nodes)
geom = eh.geometry_level_metrics(nodes)
geom_summary = eh.geometry_metrics_summary(geom)
for name, frame in [
    ('pooled_metrics', pooled),
    ('critical_bins', bins),
    ('subzone_metrics', zones),
    ('geometry_metrics', geom),
    ('geometry_summary', geom_summary),
]:
    eh.save_table(frame, RESULTS_DIR, name)

eh.plot_bin_bar(bins, 'Zonal Edge_arc_feat critical LogLife bins', FIGURES_DIR, 'critical_bins_zonal_edge_arc_feat')
eh.plot_zone_bar(zones, 'Zonal Edge_arc_feat LogLife MAE by subzone', FIGURES_DIR, 'subzone_mae_zonal_edge_arc_feat')
eh.plot_geometry_scatter(geom, FIGURES_DIR, 'zonal_edge_arc_feat_geometry')

representatives = eh.select_representative_geometries(
    geom, REGIME, ABLATION, disagreement_family='PointNetMLPJoint_FP_headfeat'
)
with open(RESULTS_DIR / 'representative_geometry_ids.json', 'w', encoding='utf-8') as stream:
    json.dump({'evaluation_label': 'validation-split evaluation', 'representatives': representatives}, stream, indent=2, default=str)
for label, sample_id in representatives.items():
    if sample_id is None:
        continue
    by_model = {
        fam: nodes[(nodes.regime == REGIME) & (nodes.ablation == ABLATION) & (nodes.sample_id == sample_id) & (nodes.model_family == fam)]
        for fam in FAMILIES
    }
    by_model = {fam: frame for fam, frame in by_model.items() if not frame.empty}
    if by_model:
        eh.plot_field_comparison(by_model, 'true_stress', 'pred_stress', 'MPa', f'{REGIME} {ABLATION} {label} Stress', mark_extrema='max', out_dir=FIGURES_DIR, filename=f'{label}_stress')
        eh.plot_field_comparison(by_model, 'true_loglife', 'pred_loglife', 'decades', f'{REGIME} {ABLATION} {label} LogLife', mark_extrema='min', out_dir=FIGURES_DIR, filename=f'{label}_loglife')
        eh.plot_arc_length_error(by_model, f'{REGIME} {ABLATION} {label}', FIGURES_DIR, f'{label}_arc_length')

def paired_family_diff(geom_df, left_family, right_family):
    records = []
    sub = geom_df[(geom_df.regime == REGIME) & (geom_df.ablation == ABLATION)]
    left = sub[sub.model_family == left_family].set_index('sample_id')
    right = sub[sub.model_family == right_family].set_index('sample_id')
    for sample_id in sorted(left.index.intersection(right.index)):
        l = left.loc[sample_id]
        r = right.loc[sample_id]
        records.append({
            'regime': REGIME,
            'ablation': ABLATION,
            'sample_id': sample_id,
            'left_family': left_family,
            'right_family': right_family,
            'left_abs_min_loglife_error': float(abs(l.min_loglife_error_decades)),
            'right_abs_min_loglife_error': float(abs(r.min_loglife_error_decades)),
            'left_minus_right_abs_min_loglife_error': float(abs(l.min_loglife_error_decades) - abs(r.min_loglife_error_decades)),
            'left_abs_max_stress_error': float(abs(l.max_stress_error)),
            'right_abs_max_stress_error': float(abs(r.max_stress_error)),
            'left_minus_right_abs_max_stress_error': float(abs(l.max_stress_error) - abs(r.max_stress_error)),
        })
    return pd.DataFrame(records)

pair_tables = {}
pair_summary_rows = []
for left_family, right_family in itertools.combinations(FAMILIES, 2):
    pair_name = f'{left_family}__vs__{right_family}'
    pair_df = paired_family_diff(geom, left_family, right_family)
    pair_tables[pair_name] = pair_df
    eh.save_table(pair_df, RESULTS_DIR, f'paired_{pair_name}')
    pair_summary_rows.append({
        'pair': pair_name,
        'left_family': left_family,
        'right_family': right_family,
        'n_geometries': len(pair_df),
        'median_left_minus_right_abs_min_loglife_error': float(pair_df.left_minus_right_abs_min_loglife_error.median()) if not pair_df.empty else np.nan,
        'fraction_right_better_min_loglife': float((pair_df.left_minus_right_abs_min_loglife_error > 0).mean()) if not pair_df.empty else np.nan,
        'median_left_minus_right_abs_max_stress_error': float(pair_df.left_minus_right_abs_max_stress_error.median()) if not pair_df.empty else np.nan,
        'fraction_right_better_max_stress': float((pair_df.left_minus_right_abs_max_stress_error > 0).mean()) if not pair_df.empty else np.nan,
    })
pair_summary = pd.DataFrame(pair_summary_rows)
eh.save_table(pair_summary, RESULTS_DIR, 'paired_summary')
display(pooled)


## Cross-notebook comparison against Notebook 1 (`01_fp_vs_argent`)

This table compares the clean **Zonal / Edge** baseline from Notebook 1 against the engineered-feature **Zonal / Edge_arc_feat** results generated here.


In [ ]:
def _read_csv_if_exists(path):
    return pd.read_csv(path) if path.exists() else None

baseline_files = {
    'pooled': _read_csv_if_exists(BASELINE_RESULTS_DIR / 'pooled_metrics.csv'),
    'bins': _read_csv_if_exists(BASELINE_RESULTS_DIR / 'critical_bins.csv'),
    'zones': _read_csv_if_exists(BASELINE_RESULTS_DIR / 'subzone_metrics.csv'),
    'geom_summary': _read_csv_if_exists(BASELINE_RESULTS_DIR / 'geometry_summary.csv'),
}
missing_baseline = [name for name, frame in baseline_files.items() if frame is None]
if missing_baseline:
    warnings.warn(
        'Notebook 1 results are missing. Run Comparison/01_fp_vs_argent first to populate: ' + ', '.join(missing_baseline),
        RuntimeWarning,
    )

family_map = {
    'ArGEnT_self_att_noSDF': 'ArGEnT_self_att_noSDF',
    'PointNetMLPJoint_headfeat': 'PointNetMLPJoint',
    'PointNetMLPJoint_FP_headfeat': 'PointNetMLPJoint_FP',
}

def scalar_value(frame, query_cols, value_col):
    if frame is None:
        return np.nan
    sub = frame.copy()
    for key, value in query_cols.items():
        sub = sub[sub[key] == value]
    if sub.empty:
        return np.nan
    return float(sub.iloc[0][value_col])

comparison_rows = []
for engineered_family, baseline_family in family_map.items():
    comparison_rows.append({
        'baseline_ablation': 'Edge',
        'engineered_ablation': ABLATION,
        'baseline_family': baseline_family,
        'engineered_family': engineered_family,
        'baseline_pooled_loglife_mae': scalar_value(baseline_files['pooled'], {'regime': REGIME, 'ablation': 'Edge', 'model_family': baseline_family, 'target': 'LogLife'}, 'MAE'),
        'engineered_pooled_loglife_mae': scalar_value(pooled, {'regime': REGIME, 'ablation': ABLATION, 'model_family': engineered_family, 'target': 'LogLife'}, 'MAE'),
        'baseline_low_life_mae': scalar_value(baseline_files['bins'], {'regime': REGIME, 'ablation': 'Edge', 'model_family': baseline_family, 'bin': 'LogLife<3'}, 'MAE'),
        'engineered_low_life_mae': scalar_value(bins, {'regime': REGIME, 'ablation': ABLATION, 'model_family': engineered_family, 'bin': 'LogLife<3'}, 'MAE'),
        'baseline_lower_transition_mae': scalar_value(baseline_files['zones'], {'regime': REGIME, 'ablation': 'Edge', 'model_family': baseline_family, 'subzone_name': 'lower_transition'}, 'MAE'),
        'engineered_lower_transition_mae': scalar_value(zones, {'regime': REGIME, 'ablation': ABLATION, 'model_family': engineered_family, 'subzone_name': 'lower_transition'}, 'MAE'),
        'baseline_median_abs_min_life_error': scalar_value(baseline_files['geom_summary'], {'regime': REGIME, 'ablation': 'Edge', 'model_family': baseline_family}, 'min_loglife_error_decades_median_abs'),
        'engineered_median_abs_min_life_error': scalar_value(geom_summary, {'regime': REGIME, 'ablation': ABLATION, 'model_family': engineered_family}, 'min_loglife_error_decades_median_abs'),
    })
comparison_table = pd.DataFrame(comparison_rows)
for stem in ['pooled_loglife_mae', 'low_life_mae', 'lower_transition_mae', 'median_abs_min_life_error']:
    comparison_table[f'delta_{stem}'] = comparison_table[f'engineered_{stem}'] - comparison_table[f'baseline_{stem}']
eh.save_table(comparison_table, RESULTS_DIR, 'cross_notebook_edge_vs_edge_arc_feat')
display(comparison_table)


## Cautious conclusion


In [ ]:
def verdict_from_pair(pair_df):
    if pair_df.empty:
        return 'no shared geometries were available for a paired conclusion.'
    median = float(pair_df['median_left_minus_right_abs_min_loglife_error'].iloc[0]) if 'median_left_minus_right_abs_min_loglife_error' in pair_df else np.nan
    frac = float(pair_df['fraction_right_better_min_loglife'].iloc[0]) if 'fraction_right_better_min_loglife' in pair_df else np.nan
    if np.isnan(median) or np.isnan(frac):
        return 'paired evidence is incomplete.'
    if median > 0 and frac > 0.5:
        return f'the right-hand model is usually better on minimum-life error (median advantage {median:.4f}; better on {frac:.1%} of shared geometries).'
    if median < 0 and frac < 0.5:
        return f'the left-hand model is usually better on minimum-life error (median advantage {abs(median):.4f}; right-hand model better on only {frac:.1%} of shared geometries).'
    return f'the paired result is mixed (median difference {median:.4f}; right-hand model better on {frac:.1%} of shared geometries).'

pair_lookup = pair_summary.set_index('pair') if not pair_summary.empty else pd.DataFrame()
argent_fp = pair_summary[pair_summary['pair'] == 'ArGEnT_self_att_noSDF__vs__PointNetMLPJoint_FP_headfeat']
regular_fp = pair_summary[pair_summary['pair'] == 'PointNetMLPJoint_headfeat__vs__PointNetMLPJoint_FP_headfeat']
change_lines = []
for _, row in comparison_table.iterrows():
    if pd.isna(row['delta_pooled_loglife_mae']):
        change_lines.append(f"- {row['engineered_family']}: baseline Notebook 1 results were unavailable, so no clean Edge delta could be computed.")
    else:
        direction = 'reduced' if row['delta_pooled_loglife_mae'] < 0 else 'increased'
        change_lines.append(
            f"- {row['engineered_family']} versus {row['baseline_family']}: engineered features {direction} pooled LogLife MAE by {abs(row['delta_pooled_loglife_mae']):.4f}, "
            f"low-life-bin MAE by {abs(row['delta_low_life_mae']):.4f}, and lower-transition MAE by {abs(row['delta_lower_transition_mae']):.4f}."
        )
summary_lines = [
    '# Engineered geometric features summary',
    '',
    f'Repository commit: `{COMMIT}`',
    '',
    'Evaluation label: **validation-split evaluation**. The same deterministic 20% geometry holdout (seed 42) is used for all models, but full independence from checkpoint selection is not proved.',
    '',
    '## Feature assignment',
    '',
    *change_lines,
    '',
    '## Paired comparisons inside Edge_arc_feat',
    '',
    f"- ArGEnT_self_att_noSDF vs PointNetMLPJoint_FP_headfeat: {verdict_from_pair(argent_fp)}",
    f"- PointNetMLPJoint_headfeat vs PointNetMLPJoint_FP_headfeat: {verdict_from_pair(regular_fp)}",
    '',
    '## Interpretation',
    '',
    'Explicit geometric features should be interpreted here as a **sensitivity / upper-bound study**: the added quantities include arc-length and derivative-derived contour features, which are cleanly available in the synthetic dataset but may be noisier or harder to recover from physical scans.',
    'If the engineered-feature deltas are small, that argues the learned models are already extracting most of the useful geometry signal from coordinates alone. If the deltas are large, they show what could be gained when high-quality derivative features are available, not guaranteed field performance on scanned hardware.',
    'The FP question should therefore be answered cautiously: use the paired Edge_arc_feat table to check whether FP still helps relative to both ArGEnT and the non-FP head-feature PointNet, but do not overstate this as independent test evidence or as proof that derivative extraction on physical scans is mature enough for deployment.',
]
summary_text = '\n'.join(summary_lines)
(RESULTS_DIR / 'comparison_summary.md').write_text(summary_text, encoding='utf-8')
display(Markdown(summary_text))
